# Object-Orientation Abusers — POO a medias

Los principios de la orientación a objetos se aplicaron de forma incorrecta o incompleta.

## Serie: Refactorización y Code Smells

Este contenido está dividido en 6 notebooks — uno por categoría de code
smell (clasificación de refactoring.guru) más un cierre de ejercicios:

1. `01_bloaters.ipynb` — Bloaters
2. **`02_object_orientation_abusers.ipynb`** — Object-Orientation Abusers
3. `03_change_preventers.ipynb` — Change Preventers
4. `04_dispensables.ipynb` — Dispensables
5. `05_couplers.ipynb` — Couplers
6. `06_ejercicios_autoevaluacion.ipynb` — Ejercicios y autoevaluación


### 1. Switch Statements (condicionales por tipo)

**Definición:** Condicionales (if/elif o switch) que ramifican según el "tipo" de un objeto, repetidos en varios lugares del código.

**Síntoma:** Cada figura/tipo nuevo obliga a editar el mismo if/elif en todos los lugares donde aparezca esa lógica.

**Técnica de refactor:** Replace Conditional with Polymorphism

#### Con el smell

In [ ]:
def calcular_area(figura):
    if figura["tipo"] == "circulo":
        return 3.1416 * figura["radio"] ** 2
    elif figura["tipo"] == "cuadrado":
        return figura["lado"] ** 2
    elif figura["tipo"] == "rectangulo":
        return figura["base"] * figura["altura"]
    raise ValueError("Figura desconocida")

figuras = [
    {"tipo": "circulo", "radio": 2},
    {"tipo": "cuadrado", "lado": 3},
]
for f in figuras:
    print(f'{f["tipo"]}: {calcular_area(f):.2f}')

#### Refactorizado

In [ ]:
class Figura:
    def area(self):
        raise NotImplementedError

class Circulo(Figura):
    def __init__(self, radio):
        self.radio = radio

    def area(self):
        return 3.1416 * self.radio ** 2

class Cuadrado(Figura):
    def __init__(self, lado):
        self.lado = lado

    def area(self):
        return self.lado ** 2

figuras = [Circulo(2), Cuadrado(3)]
for f in figuras:
    print(f"{type(f).__name__}: {f.area():.2f}")

**Explicación:** `Replace Conditional with Polymorphism` mueve cada rama del condicional a un método `area()` propio de cada subclase. Agregar una figura nueva ya no exige tocar ningún if/elif existente: solo se crea una subclase más.

### 2. Refused Bequest (legado rechazado)

**Definición:** Una subclase hereda métodos de una superclase pero no los usa — o los sobreescribe solo para lanzar una excepción.

**Síntoma:** La jerarquía de herencia no refleja una relación real "es-un": obliga a la subclase a cargar con comportamiento que no le corresponde.

**Técnica de refactor:** Replace Inheritance with Delegation / reordenar la jerarquía

#### Con el smell

In [ ]:
class Ave:
    def volar(self):
        print("Volando alto")

class Pinguino(Ave):
    def volar(self):
        # el pingüino rechaza el legado de Ave
        raise NotImplementedError("Los pingüinos no vuelan")

aves = [Ave(), Pinguino()]
for ave in aves:
    try:
        ave.volar()
    except NotImplementedError as e:
        print(f"Error: {e}")

#### Refactorizado

In [ ]:
class Ave:
    def nadar(self):
        print("Nadando")

class AveVoladora(Ave):
    def volar(self):
        print("Volando alto")

class Pinguino(Ave):
    pass  # no hereda volar(): simplemente no aplica

aves = [AveVoladora(), Pinguino()]
for ave in aves:
    ave.nadar()
    if isinstance(ave, AveVoladora):
        ave.volar()

**Explicación:** Se reordena la jerarquía: `Ave` solo contiene lo común a todas las aves (`nadar`), y `AveVoladora` agrega `volar()` solo donde aplica. `Pinguino` ya no necesita rechazar nada porque nunca heredó un método que no puede cumplir.

### 3. Temporary Field (campo temporal)

**Definición:** Un atributo de una clase que solo tiene un valor válido en ciertas circunstancias (por ejemplo, dentro de un único método) y que el resto del tiempo queda en un valor "apagado".

**Síntoma:** Un campo se inicializa, se usa en un solo método y luego hay que acordarse de "limpiarlo" — confunde a quien lea la clase, porque parece un atributo de estado permanente y no lo es.

**Técnica de refactor:** Extract Class

#### Con el smell

In [ ]:
class CalculadoraImpuestos:
    def __init__(self):
        self._descuento_temporal = 0  # solo tiene sentido durante calcular()

    def calcular(self, monto, es_vip):
        if es_vip:
            self._descuento_temporal = monto * 0.1
        total = monto - self._descuento_temporal
        self._descuento_temporal = 0  # hay que acordarse de limpiarlo
        return total

calc = CalculadoraImpuestos()
print(calc.calcular(200, True))

#### Refactorizado

In [ ]:
class DescuentoVIP:
    def calcular(self, monto, es_vip):
        return monto * 0.1 if es_vip else 0

class CalculadoraImpuestos:
    def __init__(self):
        self._descuento = DescuentoVIP()

    def calcular(self, monto, es_vip):
        descuento = self._descuento.calcular(monto, es_vip)
        return monto - descuento

calc = CalculadoraImpuestos()
print(calc.calcular(200, True))

**Explicación:** `Extract Class` mueve el cálculo del descuento (y el campo que solo tenía sentido temporalmente) a `DescuentoVIP`, una clase dedicada. `CalculadoraImpuestos` ya no necesita mantener ni limpiar un campo que solo vive durante una llamada.

### 4. Alternative Classes with Different Interfaces (clases alternativas con interfaces distintas)

**Definición:** Dos o más clases que hacen esencialmente lo mismo, pero exponen métodos con nombres o firmas distintas, impidiendo tratarlas de forma intercambiable.

**Síntoma:** El código cliente necesita saber de antemano qué clase tiene para llamar al método correcto, aunque el resultado que busca sea el mismo en ambos casos.

**Técnica de refactor:** Rename Method / Extract Superclass

#### Con el smell

In [ ]:
class ImpresoraPDF:
    def imprimir_documento(self, texto):
        print(f"[PDF] {texto}")

class ImpresoraTexto:
    def mostrar_en_pantalla(self, texto):
        print(f"[TXT] {texto}")

impresoras = [ImpresoraPDF(), ImpresoraTexto()]
impresoras[0].imprimir_documento("Hola")
impresoras[1].mostrar_en_pantalla("Hola")

#### Refactorizado

In [ ]:
class Impresora:
    def imprimir(self, texto):
        raise NotImplementedError

class ImpresoraPDF(Impresora):
    def imprimir(self, texto):
        print(f"[PDF] {texto}")

class ImpresoraTexto(Impresora):
    def imprimir(self, texto):
        print(f"[TXT] {texto}")

impresoras = [ImpresoraPDF(), ImpresoraTexto()]
for impresora in impresoras:
    impresora.imprimir("Hola")  # mismo método para todas

**Explicación:** `Rename Method` unifica el nombre del método (`imprimir`) en ambas clases, y `Extract Superclass` crea `Impresora` como interfaz común. El código cliente ahora puede recorrer cualquier lista de impresoras sin preguntar de qué tipo es cada una.